# Complete the 2026 World Cup Knockout Prediction Audit

This self-contained notebook completes MatchCast's 2026 World Cup audit for the eight
matches after the round of 16. It reproduces the four genuinely pre-kickoff
quarter-final forecasts from notebook `17`, then performs a **retrospective,
leakage-safe staged replay** for the semi-finals, bronze final, and final.

For each stage, the selected weighted logistic model is trained only on matches
completed before that stage. The later forecasts were generated on 2026-07-21, after
the tournament, so they are useful reproducibility checks rather than pre-registered
live predictions.

Actual scores and winners are transcribed from FIFA's
[official knockout-stage results](https://www.fifa.com/en/articles/knockout-stage-match-schedule-bracket).


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed" / "matches_with_elo.csv"
REPORTS = ROOT / "reports"
SEED = 20260721
LABELS = {"H": 0, "D": 1, "A": 2}
ORDERED = np.array(["H", "D", "A"])

historical = (
    pd.read_csv(DATA, parse_dates=["date"])
    .sort_values(["date", "match_id"], kind="mergesort")
    .reset_index(drop=True)
)
assert historical.date.max() == pd.Timestamp("2026-07-07")
assert len(historical[(historical.tournament == "FIFA World Cup") & (historical.date.dt.year == 2026)]) == 96

def match(match_id, date, home, away, city, home_score=None, away_score=None):
    completed = home_score is not None and away_score is not None
    result = None if not completed else ("H" if home_score > away_score else "A" if home_score < away_score else "D")
    return {
        "match_id": match_id,
        "source_row_number": -1,
        "date": pd.Timestamp(date),
        "home_team": home,
        "away_team": away,
        "original_home_team": home,
        "original_away_team": away,
        "home_score": 0 if home_score is None else home_score,
        "away_score": 0 if away_score is None else away_score,
        "tournament": "FIFA World Cup",
        "city": city,
        "country": "United States",
        "neutral": True,
        "result": "H" if result is None else result,
        "goal_difference": 0 if not completed else home_score - away_score,
    }

qf_fixtures = pd.DataFrame([
    match("WC2026_097", "2026-07-09", "France", "Morocco", "Foxborough"),
    match("WC2026_098", "2026-07-10", "Spain", "Belgium", "Inglewood"),
    match("WC2026_099", "2026-07-11", "Norway", "England", "Miami Gardens"),
    match("WC2026_100", "2026-07-11", "Argentina", "Switzerland", "Kansas City"),
])
qf_results = pd.DataFrame([
    match("WC2026_097", "2026-07-09", "France", "Morocco", "Foxborough", 2, 0),
    match("WC2026_098", "2026-07-10", "Spain", "Belgium", "Inglewood", 2, 1),
    match("WC2026_099", "2026-07-11", "Norway", "England", "Miami Gardens", 1, 2),
    match("WC2026_100", "2026-07-11", "Argentina", "Switzerland", "Kansas City", 3, 1),
])
sf_fixtures = pd.DataFrame([
    match("WC2026_101", "2026-07-14", "France", "Spain", "Arlington"),
    match("WC2026_102", "2026-07-15", "England", "Argentina", "Atlanta"),
])
sf_results = pd.DataFrame([
    match("WC2026_101", "2026-07-14", "France", "Spain", "Arlington", 0, 2),
    match("WC2026_102", "2026-07-15", "England", "Argentina", "Atlanta", 1, 2),
])
medal_fixtures = pd.DataFrame([
    match("WC2026_103", "2026-07-18", "France", "England", "Miami Gardens"),
    match("WC2026_104", "2026-07-19", "Spain", "Argentina", "East Rutherford"),
])
medal_results = pd.DataFrame([
    match("WC2026_103", "2026-07-18", "France", "England", "Miami Gardens", 4, 6),
    match("WC2026_104", "2026-07-19", "Spain", "Argentina", "East Rutherford", 1, 0),
])


## Elo replay and leakage-safe features

New result rows receive pre-match Elo values by replaying the same Elo configuration
used throughout the project. Matches on the same date are evaluated from the same
start-of-day ratings. Rolling features shift by one match, so a row never sees its own
result.


In [2]:
INITIAL_RATING = 1500.0
K_FACTOR = 20.0
DRAW_PRIOR = 0.25

def expected_score(rating_a, rating_b):
    return 1.0 / (1.0 + 10.0 ** (-(rating_a - rating_b) / 400.0))

def actual_home_score(result):
    return {"H": 1.0, "D": 0.5, "A": 0.0}[result]

def outcome_probabilities(home_elo, away_elo, neutral):
    adjusted_home = home_elo if neutral else home_elo + 100.0
    expectation = expected_score(adjusted_home, away_elo)
    return (
        (1.0 - DRAW_PRIOR) * expectation,
        DRAW_PRIOR,
        (1.0 - DRAW_PRIOR) * (1.0 - expectation),
    )

def replay_ratings(frame):
    ratings = {}
    ordered = frame.sort_values(["date", "match_id"], kind="mergesort")
    for _, day in ordered.groupby("date", sort=True):
        start = ratings.copy()
        deltas = {}
        for row in day.itertuples(index=False):
            home_elo = start.get(row.home_team, INITIAL_RATING)
            away_elo = start.get(row.away_team, INITIAL_RATING)
            adjusted_home = home_elo if bool(row.neutral) else home_elo + 100.0
            change = K_FACTOR * (actual_home_score(row.result) - expected_score(adjusted_home, away_elo))
            deltas[row.home_team] = deltas.get(row.home_team, 0.0) + change
            deltas[row.away_team] = deltas.get(row.away_team, 0.0) - change
        for team, delta in deltas.items():
            ratings[team] = start.get(team, INITIAL_RATING) + delta
    return ratings

def add_pre_match_elo(base, new_rows, update):
    ratings = replay_ratings(base)
    enriched = []
    for _, day in new_rows.sort_values(["date", "match_id"], kind="mergesort").groupby("date", sort=True):
        start = ratings.copy()
        deltas = {}
        for row in day.to_dict("records"):
            home_elo = start.get(row["home_team"], INITIAL_RATING)
            away_elo = start.get(row["away_team"], INITIAL_RATING)
            p_home, p_draw, p_away = outcome_probabilities(home_elo, away_elo, row["neutral"])
            row.update({
                "home_elo": home_elo,
                "away_elo": away_elo,
                "elo_difference": home_elo - away_elo,
                "elo_adjusted_difference": home_elo - away_elo,
                "elo_home_probability": p_home,
                "elo_draw_probability": p_draw,
                "elo_away_probability": p_away,
            })
            enriched.append(row)
            if update:
                change = K_FACTOR * (actual_home_score(row["result"]) - expected_score(home_elo, away_elo))
                deltas[row["home_team"]] = deltas.get(row["home_team"], 0.0) + change
                deltas[row["away_team"]] = deltas.get(row["away_team"], 0.0) - change
        if update:
            for team, delta in deltas.items():
                ratings[team] = start.get(team, INITIAL_RATING) + delta
    return pd.DataFrame(enriched)

def append_results(base, results):
    enriched = add_pre_match_elo(base, results, update=True)
    return (
        pd.concat([base, enriched], ignore_index=True)
        .sort_values(["date", "match_id"], kind="mergesort")
        .reset_index(drop=True)
    )

def classify_tournament(name):
    text = str(name).lower()
    if "qualif" in text:
        return "qualifier"
    if str(name) == "FIFA World Cup":
        return "world_cup"
    markers = [
        "copa america", "copa américa", "african cup of nations", "afc asian cup",
        "uefa euro", "european championship", "gold cup", "oceania", "nations league",
    ]
    if any(marker in text for marker in markers):
        return "continental"
    if text == "friendly":
        return "friendly"
    return "other"

def add_world_cup_stage(frame):
    frame = frame.copy()
    frame["wc_match_number"] = -1
    mask = frame.tournament.eq("FIFA World Cup")
    frame.loc[mask, "wc_match_number"] = frame.loc[mask].groupby(frame.loc[mask, "date"].dt.year).cumcount()
    frame["wc_group_stage"] = frame.wc_match_number.between(0, 47).astype(float)
    frame["wc_knockout_stage"] = (frame.wc_match_number >= 48).astype(float)
    return frame

def build_team_panel(frame):
    points = {"H": 1.0, "D": 0.5, "A": 0.0}
    home = pd.DataFrame({
        "match_id": frame.match_id, "date": frame.date, "team": frame.home_team, "is_home": True,
        "goals_for": frame.home_score.astype(float), "goals_against": frame.away_score.astype(float),
        "team_elo": frame.home_elo.astype(float), "result_points": frame.result.map(points).astype(float),
        "expected_points": frame.elo_home_probability + 0.5 * frame.elo_draw_probability,
    })
    away = pd.DataFrame({
        "match_id": frame.match_id, "date": frame.date, "team": frame.away_team, "is_home": False,
        "goals_for": frame.away_score.astype(float), "goals_against": frame.home_score.astype(float),
        "team_elo": frame.away_elo.astype(float), "result_points": 1.0 - frame.result.map(points).astype(float),
        "expected_points": frame.elo_away_probability + 0.5 * frame.elo_draw_probability,
    })
    panel = (
        pd.concat([home, away], ignore_index=True)
        .sort_values(["team", "date", "match_id"], kind="mergesort")
        .reset_index(drop=True)
    )
    grouped = panel.groupby("team", sort=False)
    panel["days_since_match"] = grouped.date.diff().dt.days.fillna(365).clip(0, 365)
    recent_counts = pd.Series(0.0, index=panel.index)
    for _, idx in panel.groupby("team", sort=False).groups.items():
        dates = panel.loc[idx, "date"].to_numpy()
        recent_counts.loc[idx] = [
            float(np.sum((current - dates[:pos]) <= np.timedelta64(365, "D")))
            for pos, current in enumerate(dates)
        ]
    panel["recent_matches_365"] = recent_counts
    for window in [3, 5, 10]:
        for col in ["goals_for", "goals_against", "result_points"]:
            fill = {"goals_for": 1.25, "goals_against": 1.25, "result_points": 0.5}[col]
            panel[f"rolling_{window}_{col}"] = grouped[col].transform(
                lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean()
            ).fillna(fill)
        actual = grouped.result_points.transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean())
        expected = grouped.expected_points.transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean())
        panel[f"rolling_{window}_form_vs_expected"] = (actual - expected).fillna(0.0)
        panel[f"rolling_{window}_elo_trend"] = (
            grouped.team_elo.shift(1) - grouped.team_elo.shift(1 + window)
        ).fillna(0.0)
    return panel

def build_features(frame):
    frame = add_world_cup_stage(frame)
    frame["tournament_type"] = frame.tournament.map(classify_tournament)
    panel = build_team_panel(frame)
    value_cols = [c for c in panel.columns if c.startswith("rolling_")] + ["days_since_match", "recent_matches_365"]
    home = panel.loc[panel.is_home, ["match_id", *value_cols]].rename(columns={c: f"home_{c}" for c in value_cols})
    away = panel.loc[~panel.is_home, ["match_id", *value_cols]].rename(columns={c: f"away_{c}" for c in value_cols})
    out = frame.merge(home, on="match_id", how="left").merge(away, on="match_id", how="left")
    out["abs_elo_difference"] = out.elo_difference.abs()
    out["elo_similarity"] = np.exp(-out.abs_elo_difference / 200.0)
    out["elo_difference_sq"] = np.sign(out.elo_difference) * (out.elo_difference / 400.0) ** 2
    for window in [3, 5, 10]:
        out[f"recent_goal_total_{window}"] = out[f"home_rolling_{window}_goals_for"] + out[f"away_rolling_{window}_goals_for"]
        out[f"recent_defence_total_{window}"] = out[f"home_rolling_{window}_goals_against"] + out[f"away_rolling_{window}_goals_against"]
        out[f"recent_form_gap_{window}"] = out[f"home_rolling_{window}_result_points"] - out[f"away_rolling_{window}_result_points"]
        out[f"recent_attack_gap_{window}"] = out[f"home_rolling_{window}_goals_for"] - out[f"away_rolling_{window}_goals_for"]
        out[f"recent_defence_gap_{window}"] = out[f"home_rolling_{window}_goals_against"] - out[f"away_rolling_{window}_goals_against"]
    out["rest_gap"] = out.home_days_since_match - out.away_days_since_match
    out["activity_gap_365"] = out.home_recent_matches_365 - out.away_recent_matches_365
    return out.sort_values(["date", "match_id"], kind="mergesort").reset_index(drop=True)


## Stage-by-stage model fitting

The design matrix, scaling, regularization, recency half-life, and importance weights
match notebooks `13` and `17`. A stage cutoff excludes its fixtures and every later
result from training.


In [3]:
FEATURES = [
    "elo_difference", "abs_elo_difference", "elo_difference_sq", "elo_similarity",
    "neutral", "home_elo", "away_elo", "wc_group_stage", "wc_knockout_stage",
    "home_days_since_match", "away_days_since_match", "home_recent_matches_365",
    "away_recent_matches_365", "rest_gap", "activity_gap_365",
]
for window in [3, 5, 10]:
    FEATURES += [
        f"home_rolling_{window}_goals_for", f"home_rolling_{window}_goals_against",
        f"home_rolling_{window}_result_points", f"home_rolling_{window}_form_vs_expected",
        f"home_rolling_{window}_elo_trend", f"away_rolling_{window}_goals_for",
        f"away_rolling_{window}_goals_against", f"away_rolling_{window}_result_points",
        f"away_rolling_{window}_form_vs_expected", f"away_rolling_{window}_elo_trend",
        f"recent_goal_total_{window}", f"recent_defence_total_{window}",
        f"recent_form_gap_{window}", f"recent_attack_gap_{window}",
        f"recent_defence_gap_{window}",
    ]
TOURNAMENT_LEVELS = ["world_cup", "qualifier", "continental", "friendly"]

def design(frame):
    out = frame[FEATURES].astype(float).copy()
    for col in [c for c in out.columns if "elo" in c or c in ["home_elo", "away_elo", "elo_difference", "abs_elo_difference"]]:
        out[col] = out[col] / 400.0
    for col in ["home_days_since_match", "away_days_since_match", "rest_gap"]:
        out[col] = out[col] / 30.0
    out["neutral"] = frame.neutral.astype(float)
    for level in TOURNAMENT_LEVELS:
        out[f"tournament_{level}"] = (frame.tournament_type == level).astype(float)
    return out

def sample_weights(frame):
    age_years = (frame.date.max() - frame.date).dt.days / 365.25
    recency = np.exp(-age_years / 7.0)
    importance = frame.tournament_type.map({
        "world_cup": 1.7, "qualifier": 1.25, "continental": 1.15,
        "friendly": 0.65, "other": 1.0,
    }).fillna(1.0)
    return np.asarray(recency * importance, dtype=float)

def predict_stage(base, fixtures, cutoff, stage, status):
    fixture_rows = add_pre_match_elo(base, fixtures, update=False)
    featured = build_features(pd.concat([base, fixture_rows], ignore_index=True))
    train = featured[(featured.date >= "2000-01-01") & (featured.date < pd.Timestamp(cutoff))].copy()
    target = featured[featured.match_id.isin(fixtures.match_id)].copy().sort_values(["date", "match_id"])
    assert train.date.max() < pd.Timestamp(cutoff)
    assert target.filter(regex="rolling_|days_since|recent_|elo_similarity|stage|gap").isna().sum().sum() == 0

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=0.25, max_iter=3000, random_state=SEED),
    )
    model.fit(
        design(train),
        train.result.map(LABELS).to_numpy(),
        logisticregression__sample_weight=sample_weights(train),
    )
    probs = np.asarray(model.predict_proba(design(target)), dtype=float)
    probs = probs / probs.sum(axis=1, keepdims=True)

    output = target[["match_id", "date", "home_team", "away_team", "city"]].copy()
    output["stage"] = stage
    output["forecast_status"] = status
    output["home_win_probability"] = probs[:, 0]
    output["draw_probability"] = probs[:, 1]
    output["away_win_probability"] = probs[:, 2]
    output["predicted_1x2"] = ORDERED[probs.argmax(axis=1)]
    output["model_pick"] = np.select(
        [output.predicted_1x2.eq("H"), output.predicted_1x2.eq("D")],
        [output.home_team, "Draw"],
        default=output.away_team,
    )
    return output

qf_predictions = predict_stage(
    historical, qf_fixtures, "2026-07-09", "Quarter-final", "locked pre-kickoff"
)
through_qf = append_results(historical, qf_results)
sf_predictions = predict_stage(
    through_qf, sf_fixtures, "2026-07-14", "Semi-final", "retrospective staged replay"
)
through_sf = append_results(through_qf, sf_results)
medal_predictions = predict_stage(
    through_sf, medal_fixtures, "2026-07-18", "Bronze final / final", "retrospective staged replay"
)

predictions = pd.concat([qf_predictions, sf_predictions, medal_predictions], ignore_index=True)
actuals = pd.concat([qf_results, sf_results, medal_results], ignore_index=True)[
    ["match_id", "home_score", "away_score", "result"]
].rename(columns={"result": "actual_1x2"})
audit = predictions.merge(actuals, on="match_id", validate="one_to_one")
audit["actual_winner"] = np.where(audit.actual_1x2.eq("H"), audit.home_team, audit.away_team)
audit["score"] = audit.home_score.astype(str) + "-" + audit.away_score.astype(str)
audit["correct"] = audit.predicted_1x2.eq(audit.actual_1x2)
audit[[
    "stage", "date", "home_team", "away_team", "model_pick",
    "home_win_probability", "draw_probability", "away_win_probability",
    "score", "actual_winner", "correct", "forecast_status",
]]


,stage,date,home_team,away_team,model_pick,home_win_probability,draw_probability,away_win_probability,score,actual_winner,correct,forecast_status
0,Quarter-final,2026-07-09,France,Morocco,France,0.552414,0.230319,0.217267,2-0,France,True,locked pre-kickoff
1,Quarter-final,2026-07-10,Spain,Belgium,Spain,0.544191,0.249659,0.206151,2-1,Spain,True,locked pre-kickoff
2,Quarter-final,2026-07-11,Norway,England,England,0.289645,0.232630,0.477726,1-2,England,True,locked pre-kickoff
3,Quarter-final,2026-07-11,Argentina,Switzerland,Argentina,0.670585,0.218071,0.111344,3-1,Argentina,True,locked pre-kickoff
4,Semi-final,2026-07-14,France,Spain,France,0.398287,0.263380,0.338333,0-2,Spain,False,retrospective staged replay
5,Semi-final,2026-07-15,England,Argentina,Argentina,0.268324,0.245071,0.486605,1-2,Argentina,True,retrospective staged replay
6,Bronze final / final,2026-07-18,France,England,France,0.540858,0.212624,0.246518,4-6,England,False,retrospective staged replay
7,Bronze final / final,2026-07-19,Spain,Argentina,Spain,0.414854,0.234299,0.350847,1-0,Spain,True,retrospective staged replay


In [4]:
# The four quarter-final forecasts must reproduce notebook 17 after rounding.
expected_qf = {
    "WC2026_097": ("France", 55, 23, 22),
    "WC2026_098": ("Spain", 54, 25, 21),
    "WC2026_099": ("England", 29, 23, 48),
    "WC2026_100": ("Argentina", 67, 22, 11),
}
for row in audit.iloc[:4].itertuples(index=False):
    expected = expected_qf[row.match_id]
    observed = (
        row.model_pick,
        round(100 * row.home_win_probability),
        round(100 * row.draw_probability),
        round(100 * row.away_win_probability),
    )
    assert observed == expected, (row.match_id, observed, expected)

assert len(audit) == 8
assert audit.match_id.is_unique
assert np.allclose(
    audit[["home_win_probability", "draw_probability", "away_win_probability"]].sum(axis=1),
    1.0,
)
assert audit.actual_winner.tolist() == [
    "France", "Spain", "England", "Argentina",
    "Spain", "Argentina", "England", "Spain",
]
assert audit.score.tolist() == ["2-0", "2-1", "1-2", "3-1", "0-2", "1-2", "4-6", "1-0"]

REPORTS.mkdir(exist_ok=True)
audit.to_csv(REPORTS / "world_cup_2026_remaining_predictions.csv", index=False)

lines = [
    "| Stage | Date | Match | Score | Model pick | P(home) | P(draw) | P(away) | Actual winner | Correct? |",
    "| --- | --- | --- | --- | --- | --- | --- | --- | --- | :---: |",
]
for row in audit.itertuples(index=False):
    lines.append(
        f"| {row.stage} | {row.date.date().isoformat()} | {row.home_team} vs {row.away_team} "
        f"| {row.score} | {row.model_pick} | {row.home_win_probability:.0%} "
        f"| {row.draw_probability:.0%} | {row.away_win_probability:.0%} "
        f"| {row.actual_winner} | {'✅' if row.correct else '❌'} |"
    )
table_text = "\n".join(lines)
(REPORTS / "world_cup_2026_remaining_readme_table.txt").write_text(
    table_text + "\n", encoding="utf-8"
)

print(table_text)
print(f"\nAll eight: {audit.correct.sum()}/8 correct ({audit.correct.mean():.1%})")
print(
    "Locked quarter-finals: "
    f"{audit.loc[audit.forecast_status.eq('locked pre-kickoff'), 'correct'].sum()}/4 correct"
)
print(
    "Retrospective staged replay: "
    f"{audit.loc[audit.forecast_status.ne('locked pre-kickoff'), 'correct'].sum()}/4 correct"
)


| Stage | Date | Match | Score | Model pick | P(home) | P(draw) | P(away) | Actual winner | Correct? |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | :---: |
| Quarter-final | 2026-07-09 | France vs Morocco | 2-0 | France | 55% | 23% | 22% | France | ✅ |
| Quarter-final | 2026-07-10 | Spain vs Belgium | 2-1 | Spain | 54% | 25% | 21% | Spain | ✅ |
| Quarter-final | 2026-07-11 | Norway vs England | 1-2 | England | 29% | 23% | 48% | England | ✅ |
| Quarter-final | 2026-07-11 | Argentina vs Switzerland | 3-1 | Argentina | 67% | 22% | 11% | Argentina | ✅ |
| Semi-final | 2026-07-14 | France vs Spain | 0-2 | France | 40% | 26% | 34% | Spain | ❌ |
| Semi-final | 2026-07-15 | England vs Argentina | 1-2 | Argentina | 27% | 25% | 49% | Argentina | ✅ |
| Bronze final / final | 2026-07-18 | France vs England | 4-6 | France | 54% | 21% | 25% | England | ❌ |
| Bronze final / final | 2026-07-19 | Spain vs Argentina | 1-0 | Spain | 41% | 23% | 35% | Spain | ✅ |

All eight: 6/8 correct (75.0%)

## 10,000-run Monte Carlo winner simulation

For knockout advancement, a simulated draw must still produce a winner. Draw mass is
split between the two teams in proportion to their non-draw win probabilities, then
each of the eight observed knockout fixtures is sampled 10,000 times. This conditional
fixture simulation tests whether Monte Carlo changes the modal winner picks; it does not
invent counterfactual downstream fixtures when a simulated quarter-final flips.


In [5]:
SIMULATIONS = 10_000
SIMULATION_SEEDS = list(range(20260721, 20260731))

def simulate_fixture_winners(frame, simulations, seed):
    rng = np.random.default_rng(seed)
    rows = []
    for row in frame.itertuples(index=False):
        decisive = row.home_win_probability + row.away_win_probability
        home_advancement_probability = (
            row.home_win_probability
            + row.draw_probability * row.home_win_probability / decisive
        )
        home_wins = int((rng.random(simulations) < home_advancement_probability).sum())
        away_wins = simulations - home_wins
        simulation_pick = row.home_team if home_wins >= away_wins else row.away_team
        rows.append({
            'match_id': row.match_id,
            'simulated_home_wins': home_wins,
            'simulated_away_wins': away_wins,
            'simulated_home_share': home_wins / simulations,
            'simulated_away_share': away_wins / simulations,
            'simulation_pick': simulation_pick,
        })
    return pd.DataFrame(rows)

primary_simulation = simulate_fixture_winners(audit, SIMULATIONS, SIMULATION_SEEDS[0])
simulation_audit = audit.merge(primary_simulation, on='match_id', validate='one_to_one')
simulation_audit['simulation_correct'] = simulation_audit.simulation_pick.eq(simulation_audit.actual_winner)

rerun_records = []
for seed in SIMULATION_SEEDS:
    rerun = simulate_fixture_winners(audit, SIMULATIONS, seed)
    compared = audit[['match_id', 'actual_winner']].merge(rerun, on='match_id')
    rerun_records.append({
        'seed': seed,
        'correct': int(compared.simulation_pick.eq(compared.actual_winner).sum()),
        'picks': tuple(compared.simulation_pick),
    })
reruns = pd.DataFrame(rerun_records)

direct_accuracy = int(audit.correct.sum())
simulation_accuracy = int(simulation_audit.simulation_correct.sum())
assert simulation_audit.simulation_pick.tolist() == audit.model_pick.tolist()
assert simulation_accuracy == direct_accuracy == 6
assert reruns.correct.eq(6).all()
assert reruns.picks.nunique() == 1

simulation_lines = [
    '| Match | Model pick | 10k simulated winner | Simulated share | Actual winner | Correct? |',
    '| --- | --- | --- | --- | --- | :---: |',
]
for row in simulation_audit.itertuples(index=False):
    winner_share = row.simulated_home_share if row.simulation_pick == row.home_team else row.simulated_away_share
    simulation_lines.append(
        f"| {row.home_team} vs {row.away_team} | {row.model_pick} | {row.simulation_pick} "
        f"| {winner_share:.1%} | {row.actual_winner} | {'✅' if row.simulation_correct else '❌'} |"
    )
simulation_table = '\n'.join(simulation_lines)
(REPORTS / 'world_cup_2026_monte_carlo_readme_table.txt').write_text(
    simulation_table + '\n', encoding='utf-8'
)

print(simulation_table)
print(f'\nDirect model accuracy: {direct_accuracy}/8')
print(f'10,000-run modal-pick accuracy: {simulation_accuracy}/8')
print(f'Reruns: {len(reruns)} seeds, all {reruns.correct.iloc[0]}/8 with identical modal picks')
print('Accuracy increased: no')


| Match | Model pick | 10k simulated winner | Simulated share | Actual winner | Correct? |
| --- | --- | --- | --- | --- | :---: |
| France vs Morocco | France | France | 72.3% | France | ✅ |
| Spain vs Belgium | Spain | Spain | 73.3% | Spain | ✅ |
| Norway vs England | England | England | 62.4% | England | ✅ |
| Argentina vs Switzerland | Argentina | Argentina | 85.5% | Argentina | ✅ |
| France vs Spain | France | France | 53.1% | Spain | ❌ |
| England vs Argentina | Argentina | Argentina | 64.6% | Argentina | ✅ |
| France vs England | France | France | 69.0% | England | ❌ |
| Spain vs Argentina | Spain | Spain | 53.3% | Spain | ✅ |

Direct model accuracy: 6/8
10,000-run modal-pick accuracy: 6/8
Reruns: 10 seeds, all 6/8 with identical modal picks
Accuracy increased: no


## Interpretation

The locked quarter-final block may be added to the live audit because it was committed
before kickoff. The four later predictions must remain labelled retrospective: their
feature construction is leakage-safe, but they were generated after the actual matches
had been played. The `actual_winner` column records advancement/winning teams, while
the probability columns remain three-way match-result probabilities. The 10,000-run
Monte Carlo layer quantifies winner uncertainty but does not increase accuracy: its modal
picks are identical to the direct model picks across every tested seed.
